In [ ]:
import sys
sys.path.append('/opt/workspace')

from datetime import datetime, timedelta
from connectors.clickhouse_client import ClickHouseClient

In [ ]:
ref_date = datetime.now().strftime("%Y-%m-%d")
previous_date = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")

ref_date, previous_date


In [ ]:
client = ClickHouseClient()


In [ ]:
table_name = "YOUR_TABLE_NAME"


In [ ]:
inserts_query = f"""
INSERT INTO silver.delta_events (
    ref_date, table_name, primary_key, operation_type, 
    row_hash_after, data_after
)
SELECT
    toDate('{ref_date}') AS ref_date,
    '{table_name}' AS table_name,
    current.primary_key,
    'INSERT' AS operation_type,
    current.row_hash AS row_hash_after,
    current.data AS data_after
FROM bronze.snapshot_raw AS current
LEFT JOIN bronze.snapshot_raw AS previous
    ON current.primary_key = previous.primary_key
    AND current.table_name = previous.table_name
    AND previous.ref_date = toDate('{previous_date}')
WHERE current.ref_date = toDate('{ref_date}')
    AND current.table_name = '{table_name}'
    AND previous.primary_key IS NULL
"""

client.execute_query(inserts_query)


In [ ]:
inserts_count = client.execute_query_with_result(
    f"""
    SELECT count() as total
    FROM silver.delta_events
    WHERE ref_date = toDate('{ref_date}')
        AND table_name = '{table_name}'
        AND operation_type = 'INSERT'
    """
)
inserts_count.result_rows


In [ ]:
updates_query = f"""
INSERT INTO silver.delta_events (
    ref_date, table_name, primary_key, operation_type,
    row_hash_before, row_hash_after, data_before, data_after
)
SELECT
    toDate('{ref_date}') AS ref_date,
    '{table_name}' AS table_name,
    current.primary_key,
    'UPDATE' AS operation_type,
    previous.row_hash AS row_hash_before,
    current.row_hash AS row_hash_after,
    previous.data AS data_before,
    current.data AS data_after
FROM bronze.snapshot_raw AS current
INNER JOIN bronze.snapshot_raw AS previous
    ON current.primary_key = previous.primary_key
    AND current.table_name = previous.table_name
    AND previous.ref_date = toDate('{previous_date}')
WHERE current.ref_date = toDate('{ref_date}')
    AND current.table_name = '{table_name}'
    AND current.row_hash != previous.row_hash
"""

client.execute_query(updates_query)


In [ ]:
updates_count = client.execute_query_with_result(
    f"""
    SELECT count() as total
    FROM silver.delta_events
    WHERE ref_date = toDate('{ref_date}')
        AND table_name = '{table_name}'
        AND operation_type = 'UPDATE'
    """
)
updates_count.result_rows


In [ ]:
deletes_query = f"""
INSERT INTO silver.delta_events (
    ref_date, table_name, primary_key, operation_type,
    row_hash_before, data_before
)
SELECT
    toDate('{ref_date}') AS ref_date,
    '{table_name}' AS table_name,
    previous.primary_key,
    'DELETE' AS operation_type,
    previous.row_hash AS row_hash_before,
    previous.data AS data_before
FROM bronze.snapshot_raw AS previous
LEFT JOIN bronze.snapshot_raw AS current
    ON previous.primary_key = current.primary_key
    AND previous.table_name = current.table_name
    AND current.ref_date = toDate('{ref_date}')
WHERE previous.ref_date = toDate('{previous_date}')
    AND previous.table_name = '{table_name}'
    AND current.primary_key IS NULL
"""

client.execute_query(deletes_query)


In [ ]:
deletes_count = client.execute_query_with_result(
    f"""
    SELECT count() as total
    FROM silver.delta_events
    WHERE ref_date = toDate('{ref_date}')
        AND table_name = '{table_name}'
        AND operation_type = 'DELETE'
    """
)
deletes_count.result_rows


In [ ]:
update_state_query = f"""
INSERT INTO silver.current_state (
    table_name, primary_key, row_hash, data,
    first_seen_date, last_seen_date, is_active
)
SELECT
    table_name,
    primary_key,
    row_hash,
    data,
    ref_date AS first_seen_date,
    ref_date AS last_seen_date,
    1 AS is_active
FROM bronze.snapshot_raw
WHERE ref_date = toDate('{ref_date}')
    AND table_name = '{table_name}'
"""

client.execute_query(update_state_query)


In [ ]:
summary = client.execute_query_with_result(
    f"""
    SELECT
        operation_type,
        count() as total
    FROM silver.delta_events
    WHERE ref_date = toDate('{ref_date}')
        AND table_name = '{table_name}'
    GROUP BY operation_type
    """
)
summary.result_rows


In [ ]:
client.close()
